In [1]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
matplotlib.use('Cairo')

In [2]:
world = gpd.read_file('./data/shp/territory_vis.shp')

In [3]:
hp_pnt = pd.read_csv('./data/hydropower/hp_attr_merit_pnt_2024.csv')

hp_pnt = gpd.GeoDataFrame(hp_pnt,geometry=gpd.points_from_xy(x=hp_pnt['lon'],
                                                             y=hp_pnt['lat']),crs='epsg:4326')


In [4]:
hp_cap_grid = pd.read_csv('./data/hydropower/hp_tot_cap_grid.csv')

In [5]:
hp_ele = pd.read_csv('./data/hydropower/hp_tot_ele_grid.csv')
grid = gpd.read_file('./data/shp/grid_vis.shp')

hp_ele_grid = pd.merge(left=grid,right=hp_ele,on=['region','grid'])

In [6]:
def add_colorbar(fig,ax,vmax,label,vmin=0,cmap='autumn_r',loc=[0.175,0.39,0.02,0.2]):
        
    cax = ax.inset_axes(loc,transform=ax.transAxes)
    
    im = plt.cm.ScalarMappable(cmap=cmap,
                               norm=plt.Normalize(vmin=vmin,
                                                  vmax=vmax))
    
    cbar = fig.colorbar(im,cax=cax,extend='max')
    
    cbar.outline.set_edgecolor('white')
    
    cbar.dividers.set_color('red')
    
    cbar.ax.tick_params(labelsize=15) 
    
    cbar.minorticks_on()
    
    cax.yaxis.tick_left()
    
    cbar.set_label(label,font={'size':25})
    


def draw_attr_map(fig,ax,geo,col,cmap,vmax,cbar_label,cbar_loc,size=None,draw_edge=False,crs='epsg:4326'):
    if draw_edge:
        geo.to_crs(crs).plot(ax=ax,
                             column=col,
                             cmap=cmap,
                             vmax=vmax,
                             edgecolor='white',
                             linewidth=0.5)
    else:
        geo.to_crs(crs).plot(ax=ax,
                column=col,
                cmap=cmap,
                vmax=vmax,
                markersize=size)
        
    
    add_colorbar(fig=fig,
                 ax=ax,
                 label=cbar_label,
                 vmax=vmax,
                 cmap=cmap,
                 loc=cbar_loc)
    
    

In [7]:
hp_pnt['size'] = 1 * hp_pnt['tot_cap']

hp_pnt['tot_cap'] = 1000 * hp_pnt['tot_cap']

In [8]:
hp_ele_grid['ele'] = hp_ele_grid['ele'] * 1e-3

In [9]:
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 25

crs = 'epsg:4326'

fig,ax = plt.subplots(figsize=(24,10),dpi=600)

world.to_crs(crs).plot(ax=ax,
                       facecolor='gainsboro',
                       edgecolor='silver',
                       alpha=0.2)

draw_attr_map(fig=fig,
              ax=ax,
              geo=hp_ele_grid,
              cmap='GnBu',
              col='ele',
              vmax=500,
              draw_edge=True,
              cbar_label='Electricity (TWh/yr)',
              cbar_loc=[0.95,0.275,0.02,0.4],
              crs=crs)

draw_attr_map(fig=fig,
              ax=ax,
              geo=hp_pnt,
              cmap='rainbow_r',
              col='tot_cap',
              vmax=1000,
              cbar_label='Capacity (MW)',
              size='size',
              cbar_loc=[0.05,0.275,0.02,0.4],
              crs=crs)

handles, labels = ax.get_legend_handles_labels()
            
handles.append(plt.Line2D([0], [0], 
                          marker='o', 
                          color='w', 
                          markerfacecolor='#FD3118',
                          markeredgecolor='black', 
                          markersize=25))

labels.append('Hydropower site')

ax.legend(handles, labels,
          loc='lower left',
          bbox_to_anchor=(0.0345, 0.18),
          frameon=False,
          ncol=1,
          prop={'size':20})

ax.axis('off')

ax.set_xticks([],[])
ax.set_yticks([],[])

plt.savefig('./fig/hp_cap.jpg',bbox_inches='tight')

plt.close()